# Paperclip Literature Retrieval

![Paperclip Literature Retrieval](https://proto-bio.github.io/proto-assets/images/tool/paperclip/hero.png)

This notebook demonstrates the four Paperclip tools, which search and read biomedical literature through GXL's hosted Paperclip service. `run_paperclip_search` finds papers by topic, `run_paperclip_lookup` finds papers by an identifier such as a DOI or PMID, `run_paperclip_grep` finds every paper whose full text mentions an exact term, and `run_paperclip_document` reads part of one paper. Together they follow the workflow GXL recommends: find the relevant papers first, then read only the lines that matter and cite them by line number. Set `PAPERCLIP_API_KEY` in your environment before running.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("paperclip")
display_overview("paperclip")
display_docs_section("paperclip", "Background")

## Available tools

In [ ]:
display_available_tools("paperclip")

### `run_paperclip_search`

`run_paperclip_search` is the starting point for most literature questions. It takes a free-text query and ranks papers by a mix of keyword and meaning-based matching, so relevant work surfaces even when it uses different wording. Each hit carries its identifier, citation details, abstract, the passage that matched, and a relevance score. By default it searches PubMed Central, which GXL recommends for general biomedical literature.

In [ ]:
from proto_tools import (
    PaperclipSearchConfig,
    PaperclipSearchInput,
    run_paperclip_search,
)

In [ ]:
# Display docs
display_api_reference("paperclip", "input", "run_paperclip_search")

# Search for published work on designing binders for CAR T cells
inputs = PaperclipSearchInput(query="CAR T cell binder design")

In [ ]:
# Display config docs
display_api_reference("paperclip", "config", "run_paperclip_search")

# Return the top 5 hits from PubMed Central | see docs above for all fields
config = PaperclipSearchConfig(limit=5)

In [ ]:
# Run the search
search_output = run_paperclip_search(inputs, config)

In [ ]:
# Display output docs
display_api_reference("paperclip", "output", "run_paperclip_search")

# Inspect the ranked hits
for hit in search_output.hits:
    print(f"{hit.document_id}  ({hit.year})  {hit.title[:80]}")
    if hit.matched_text:
        print(f"    matched: {hit.matched_text[:120]}...")

#### Filtering by Corpus and Year

Metadata filters narrow a search more reliably than adding those words to the query. Here we search all four full-text corpora at once, including bioRxiv, medRxiv, and arXiv preprints, and keep only papers published in 2025.

In [ ]:
# Search PMC and all preprint servers, restricted to 2025
recent_output = run_paperclip_search(
    PaperclipSearchInput(query="de novo protein binder design"),
    PaperclipSearchConfig(sources=["papers"], year=2025, limit=5),
)

for hit in recent_output.hits:
    print(f"{hit.document_id}  [{hit.source}]  {hit.title[:80]}")

### `run_paperclip_lookup`

When you already know which paper you want, `run_paperclip_lookup` finds it by a single metadata field: a DOI, PMID, PMC ID, arXiv ID, title, author, journal, or year. This bridges identifiers produced by other tools, such as a PMID from NCBI or a DOI cited in a UniProt entry, to the Paperclip identifier needed to read the paper.

In [ ]:
from proto_tools import (
    PaperclipLookupConfig,
    PaperclipLookupInput,
    run_paperclip_lookup,
)

In [ ]:
# Display docs
display_api_reference("paperclip", "input", "run_paperclip_lookup")

# Look up a paper by its PubMed Central ID
lookup_inputs = PaperclipLookupInput(field="pmc", value="PMC11754230")

In [ ]:
# Display config docs
display_api_reference("paperclip", "config", "run_paperclip_lookup")

# Use default config | see docs above for all fields
lookup_config = PaperclipLookupConfig()

In [ ]:
# Run the lookup
lookup_output = run_paperclip_lookup(lookup_inputs, lookup_config)

In [ ]:
# Display output docs
display_api_reference("paperclip", "output", "run_paperclip_lookup")

# Inspect the matching paper
for hit in lookup_output.hits:
    print(f"Document ID: {hit.document_id}")
    print(f"Title: {hit.title}")
    print(f"Journal: {hit.journal} ({hit.year})")
    print(f"DOI: {hit.doi}  PMID: {hit.pmid}")

### `run_paperclip_grep`

`run_paperclip_grep` searches the full text of every paper for a regular expression. Unlike search, which ranks papers by relevance, grep finds every paper that mentions an exact term, including mentions buried in methods or data-availability sections. This makes it the right tool for tracing where a specific gene, tool, dataset, or accession has been used. Because the scan is time-bounded, repeated calls can return a different set or order of matches.

In [ ]:
from proto_tools import (
    PaperclipGrepConfig,
    PaperclipGrepInput,
    run_paperclip_grep,
)

In [ ]:
# Display docs
display_api_reference("paperclip", "input", "run_paperclip_grep")

# Find papers whose full text mentions ProteinMPNN
grep_inputs = PaperclipGrepInput(pattern="ProteinMPNN")

In [ ]:
# Display config docs
display_api_reference("paperclip", "config", "run_paperclip_grep")

# Return up to 20 matching lines | see docs above for all fields
grep_config = PaperclipGrepConfig(max_matches=20)

In [ ]:
# Run the grep
grep_output = run_paperclip_grep(grep_inputs, grep_config)

In [ ]:
# Display output docs
display_api_reference("paperclip", "output", "run_paperclip_grep")

# Inspect the papers and lines that matched
print(f"Papers with a match: {len(grep_output.document_ids)}")
print(grep_output.document_ids[:10])
for line in grep_output.lines[:5]:
    print(line[:120])

### `run_paperclip_document`

`run_paperclip_document` reads one paper. It always returns the paper's metadata, and by default also reads its first 100 lines, which usually cover the title and abstract. Every line of a Paperclip paper is numbered, so evidence can be cited by line. GXL recommends reading a range of lines or the lines that match a pattern rather than the whole paper; full text is available with `read="full_text"` when you really need it.

In [ ]:
from proto_tools import (
    PaperclipDocumentConfig,
    PaperclipDocumentInput,
    run_paperclip_document,
)

In [ ]:
# Display docs
display_api_reference("paperclip", "input", "run_paperclip_document")

# Read the paper found by the lookup above
document_inputs = PaperclipDocumentInput(document_id="PMC11754230")

In [ ]:
# Display config docs
display_api_reference("paperclip", "config", "run_paperclip_document")

# Read the first 40 lines | see docs above for all fields
document_config = PaperclipDocumentConfig(read="lines", start_line=1, end_line=40)

In [ ]:
# Run the read
document_output = run_paperclip_document(document_inputs, document_config)

In [ ]:
# Display output docs
display_api_reference("paperclip", "output", "run_paperclip_document")

# Inspect the metadata and the excerpt
document = document_output.document
print(f"Title: {document.metadata.get('title')}")
print(f"Access kind: {document.access_kind}")
print(document.excerpt[:800] if document.excerpt else "No text returned")

#### Reading the Lines That Match a Pattern

Setting `read="grep"` returns only the lines that match a pattern, each with its line number, plus optional surrounding context lines. This is the most efficient way to pull a specific measurement or claim out of a long paper, and the line numbers go straight into a citation.

In [ ]:
# Find every mention of binding affinity, with one line of context on each side
affinity_output = run_paperclip_document(
    document_inputs,
    PaperclipDocumentConfig(read="grep", pattern="affinity|K_?[dD]", ignore_case=True, context_lines=1),
)

for match in affinity_output.document.matches:
    marker = ">" if match.is_match else " "
    print(f"{marker} L{match.line_number}: {match.text[:120]}")

#### From Search to Evidence

The tools are designed to chain. Here we take the top hit from the first search and pull the lines from it that mention CAR T cells, producing line-numbered evidence that can be cited directly.

In [ ]:
top_hit = search_output.hits[0]
evidence = run_paperclip_document(
    PaperclipDocumentInput(document_id=top_hit.document_id),
    PaperclipDocumentConfig(read="grep", pattern="CAR[- ]?T", ignore_case=True),
).document

print(f"{top_hit.title}\n")
for match in evidence.matches[:10]:
    print(f"L{match.line_number}: {match.text[:120]}")

## Export Results

Search results and documents can be saved to JSON, including their source provenance, for downstream use in other tools or analysis pipelines.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

search_output.export("paperclip_search", export_path=output_dir, file_format="json")
document_output.export("paperclip_document", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir}")